# Stage 3 Revised — Berth State, Berth Entry, and Predicted Release Inputs

Tujuan tahap ini adalah menyediakan keadaan operasional yang diperlukan Stage 4:
waktu mulai sandar, lama sandar berjalan, perkiraan sisa pelayanan, dan perkiraan
waktu dermaga dilepas oleh kapal yang sedang menempatinya.

In [ ]:
#@title Shared MFAR paths and stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Locate the code repository only; all simulation I/O paths are resolved by
# src.mfar_paths through MFAR_GDRIVE_ROOT or the mounted/synchronized Drive.
_code_candidates = [Path.cwd(), Path.cwd().parent]
if os.environ.get("MFAR_CODE_ROOT"):
    _code_candidates.insert(0, Path(os.environ["MFAR_CODE_ROOT"]))
_code_candidates.extend([
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
])
for _candidate in _code_candidates:
    if (_candidate / "src" / "mfar_paths.py").is_file():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "Modul src/mfar_paths.py tidak ditemukan. Jalankan notebook dari repository "
        "atau tetapkan MFAR_CODE_ROOT ke folder repository."
    )

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, DATA_RAW_DIR, CONFIG_DIR,
    STAGE_OUTPUT_DIR, STAGE_01_DIR, STAGE_02_DIR, STAGE_03_DIR,
    STAGE_04_DIR, STAGE_05_DIR, STAGE_06_DIR, STAGE_07_DIR,
    validate_csv_input, validate_raw_inputs, validate_writable_directory,
    write_execution_metadata,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)

NOTEBOOK_NAME = "03_Monitoring_State.ipynb"
RAW, CFG, STAGE = DATA_RAW_DIR, CONFIG_DIR, STAGE_OUTPUT_DIR
validate_writable_directory(STAGE_03_DIR, NOTEBOOK_NAME, 3)
print("Input stage:", STAGE_02_DIR)
print("Output folder:", STAGE_03_DIR)


In [ ]:
INFILE = STAGE/"stage_02"/"02_vessel_interpolated_grid.csv"
if not INFILE.exists():
    raise FileNotFoundError(
        f"Output Stage 2 tidak ditemukan: {INFILE}. Jalankan Stage 2 terlebih dahulu."
    )

state = validate_csv_input(INFILE, ["grid_time","mmsi","operational_status","current_berth_id","is_at_berth"], NOTEBOOK_NAME, 3)
state["grid_time"] = pd.to_datetime(state["grid_time"], errors="coerce")
state = state.dropna(subset=["grid_time","mmsi"]).sort_values(["mmsi","grid_time"])

profiles = pd.read_csv(CFG/"vessel_profiles.csv")
berths = pd.read_csv(CFG/"terminal_berths.csv")

for col in ["turnaround_min","approach_allowance_min","vehicle_capacity_ce","normal_sog_kn"]:
    if col not in state.columns:
        state = state.merge(profiles[["mmsi",col]], on="mmsi", how="left")

In [ ]:
# Normalisasi status sandar.
state["is_at_berth"] = state["operational_status"].astype(str).str.startswith("AT_BERTH")
state["berth_episode_start"] = False

for mmsi, idx in state.groupby("mmsi").groups.items():
    idx = list(idx)
    prev = False
    for i in idx:
        now = bool(state.loc[i,"is_at_berth"])
        state.loc[i,"berth_episode_start"] = now and not prev
        prev = now

# Nomor episode sandar per kapal.
state["berth_episode_id"] = (
    state.groupby("mmsi")["berth_episode_start"].cumsum()
)

episode_start = (
    state[state["is_at_berth"]]
    .groupby(["mmsi","berth_episode_id"])["grid_time"]
    .transform("min")
)
state.loc[state["is_at_berth"],"berth_entry_time"] = episode_start
state["berth_entry_time"] = pd.to_datetime(state["berth_entry_time"], errors="coerce")

state["elapsed_berth_min"] = np.where(
    state["is_at_berth"],
    (state["grid_time"]-state["berth_entry_time"]).dt.total_seconds()/60,
    0.0
)

# turnaround_min dipakai sebagai estimasi total pelayanan di dermaga.
state["predicted_remaining_service_min"] = np.where(
    state["is_at_berth"],
    np.maximum(
        0.0,
        pd.to_numeric(state["turnaround_min"], errors="coerce").fillna(40.0)
        - state["elapsed_berth_min"]
    ),
    0.0
)

# Tambahkan allowance manuver keluar 5 menit.
DEPARTURE_MANEUVER_MIN = 5.0
state["predicted_berth_release_time"] = pd.NaT
mask = state["is_at_berth"]
state.loc[mask,"predicted_berth_release_time"] = (
    state.loc[mask,"grid_time"]
    + pd.to_timedelta(
        state.loc[mask,"predicted_remaining_service_min"] + DEPARTURE_MANEUVER_MIN,
        unit="m"
    )
)

# Dermaga yang sedang ditempati.
state["occupied_berth_id"] = np.where(
    state["is_at_berth"],
    state["current_berth_id"],
    np.nan
)

In [ ]:
S3 = STAGE/"stage_03"
state.to_csv(S3/"03_input_state_enhanced.csv", index=False)

release = state.loc[state["is_at_berth"],[
    "grid_time","mmsi","vessel_name","origin","occupied_berth_id",
    "berth_entry_time","elapsed_berth_min","predicted_remaining_service_min",
    "predicted_berth_release_time","turnaround_min"
]].copy()
release.to_csv(S3/"03_predicted_berth_release_state.csv", index=False)

audit = pd.DataFrame({
    "check":[
        "missing_release_time_for_occupied_berth",
        "negative_elapsed_berth",
        "negative_remaining_service",
        "duplicate_vessel_time"
    ],
    "failed_rows":[
        int(state.loc[mask,"predicted_berth_release_time"].isna().sum()),
        int((state["elapsed_berth_min"]<0).sum()),
        int((state["predicted_remaining_service_min"]<0).sum()),
        int(state.duplicated(["grid_time","mmsi"]).sum())
    ]
})
audit.to_csv(S3/"03_berth_prediction_audit.csv", index=False)
display(audit)

## Keluaran interpretatif otomatis\n\nCSV/JSON dipertahankan untuk kontrak data. Sel berikut membuat keluaran yang dapat dibaca dan dieksplorasi tanpa membuka CSV mentah.

In [ ]:
#@title Export readable Stage 3 outputs
from src.mfar_visuals import stage3_berth_outputs
_readable_outputs = stage3_berth_outputs(
    state, audit, STAGE_03_DIR
)
print("Readable Stage 3 outputs:")
for _path in _readable_outputs:
    print("-", _path.name)


In [ ]:
#@title Execution metadata and saved-artifact report
_stage_dir = STAGE_03_DIR
_saved_files = sorted(_stage_dir.glob("03_*"))
write_execution_metadata(
    stage=3, notebook=NOTEBOOK_NAME, started_at=_MFAR_STARTED_AT,
    input_paths=[INFILE, CFG/'vessel_profiles.csv', CFG/'terminal_berths.csv'],
    input_rows={"state": len(state)},
    output_rows={"state": len(state)},
    output_files=_saved_files,
)
